# Intermediate: Convolutional Neural Networks

This notebook covers:
- Convolution operations in depth
- Pooling strategies
- Famous CNN architectures (LeNet, AlexNet, VGG, ResNet basics)
- Image classification end-to-end
- Feature visualization

## Convolution Operation

```
Input (3x3)    Kernel (2x2)    Output (2x2)
[1 2 3]        [1 0]           [1*1+2*0+4*1+5*0  2*1+3*0+5*1+6*0]
[4 5 6]  *     [1 0]     =     [4*1+5*0+7*1+8*0  5*1+6*0+8*1+9*0]
[7 8 9]
                                = [5  7]
                                  [11 13]
```


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


## 1. Understanding Convolutions


In [ ]:
# Manual convolution example
input_tensor = torch.tensor([[
    [1., 2., 3., 4.],
    [5., 6., 7., 8.],
    [9., 10., 11., 12.],
    [13., 14., 15., 16.]
]]).unsqueeze(0)  # Add batch dimension

print(f"Input shape: {input_tensor.shape}")

# Define a simple kernel (edge detector)
conv = nn.Conv2d(1, 1, kernel_size=3, padding=0, bias=False)
with torch.no_grad():
    conv.weight = nn.Parameter(torch.tensor([[[
        [-1., -1., -1.],
        [0., 0., 0.],
        [1., 1., 1.]
    ]]]))

output = conv(input_tensor)
print(f"Output shape: {output.shape}")
print(f"Output:\n{output.squeeze()}")


## 2. LeNet-5 Architecture

One of the first successful CNNs (1998).

```
LeNet-5 Architecture:
Input (28x28) → Conv(6,5x5) → Pool(2x2) → Conv(16,5x5) → Pool(2x2) → FC(120) → FC(84) → FC(10)
```


In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*4*4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)
    
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, 16*4*4)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = LeNet5().to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


## 3. Modern CNN Architecture

Building a VGG-style network for CIFAR-10.


In [ ]:
class VGGBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_convs):
        super(VGGBlock, self).__init__()
        layers = []
        for _ in range(num_convs):
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
            layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU(inplace=True))
            in_channels = out_channels
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.block = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.block(x)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            VGGBlock(3, 64, 2),
            VGGBlock(64, 128, 2),
            VGGBlock(128, 256, 3),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


## 4. Training on CIFAR-10


In [ ]:
# Data preparation
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10('./data', train=False, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")


## 5. ResNet Basics: Skip Connections

```
ResNet Block:
       x
       |
      Conv
       |
      ReLU
       |
      Conv
       |
       +  ← skip connection (x)
       |
      ReLU
       |
      out
```


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)  # Skip connection
        out = F.relu(out)
        return out

# Test residual block
block = ResidualBlock(64, 64).to(device)
x = torch.randn(1, 64, 32, 32).to(device)
out = block(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")


## 📝 Summary

✅ Understood convolution operations  
✅ Built LeNet-5 architecture  
✅ Created VGG-style networks  
✅ Implemented ResNet skip connections  
✅ Trained models on CIFAR-10  

### Next Steps

- **Next**: `02_rnn.ipynb` - Recurrent Neural Networks
- **Practice**: Implement other CNN architectures
- **Challenge**: Train on ImageNet subset
